# Observed VoiceText tree feature values

Explore the captured inputs and outputs at the `tree3` lookup boundary. This notebook reuses the repository's parser and runtime comparator, then summarizes how much each raw selector varied in the captured Paul voice-tree calls. It is an exploration aid: selector numbers and output values remain unnamed, and capture counts do not establish phonetic meaning.

Run from the repository root. The notebook needs the checked-in Stage 6 lookup logs plus locally available `data-common/dict-eng` and `data-paul/M16/ttsdata/tree3` assets. It reads those assets without modifying or embedding them. See the Markdown [Stage 5 findings](../voice-engine-and-model-formats.md#stage-5-tree3-parser-and-caller-behavior) for the authoritative interpretation and evidence limits.

In [ ]:
from collections import defaultdict
from html import escape
from pathlib import Path
import sys

from IPython.display import HTML, SVG, display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "tools/revkit/scripts/compare_tree3_runtime.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Could not find the repository root above the current directory.")
SCRIPT_DIR = ROOT / "tools/revkit/scripts"
sys.path.insert(0, str(SCRIPT_DIR))
from compare_tree3_runtime import load_trees, parse_lookup

LOGS = {
    "ordinary": ROOT / "tools/revkit/work/stage5/probes/stage6/ordinary/tree-lookups-gdb.log",
    "numbers": ROOT / "tools/revkit/work/stage5/probes/stage6/numbers/tree-lookups-gdb.log",
    "abbreviations": ROOT / "tools/revkit/work/stage5/probes/stage6/abbreviations/tree-lookups-gdb.log",
}
missing_logs = [str(path) for path in LOGS.values() if not path.is_file()]
if missing_logs:
    raise FileNotFoundError("Missing tracked runtime log(s): " + ", ".join(missing_logs))

trees = load_trees(ROOT)
trees_by_shape = defaultdict(list)
for tree in trees:
    trees_by_shape[(len(tree.nodes), tree.output_width)].append(tree)

print(f"Loaded {len(trees)} tree resources and {len(LOGS)} runtime captures.")

## Match runtime lookups to tree resources

Each trace records the input vector and the DLL's returned scalar or vector. The existing comparator evaluates all same-shape local trees and accepts a match only when exactly one predicts the captured result. The following cell retains those uniquely matched records and reports any ambiguous or unmatched calls.

In [ ]:
lookups = []
unmatched = []
for probe, path in LOGS.items():
    lines = path.read_text(errors="replace").splitlines()
    for index, line in enumerate(lines):
        if not (line.startswith("SCALAR_ENTRY") or line.startswith("VECTOR_ENTRY")):
            continue
        features, actual, shape = parse_lookup(lines, index)
        candidates = trees_by_shape.get(shape, [])
        matches = []
        for tree in candidates:
            try:
                _, expected = tree.evaluate(list(features))
            except ValueError:
                continue
            if expected == actual:
                matches.append(tree)
        if len(matches) != 1:
            unmatched.append({"probe": probe, "matches": len(matches), "shape": shape})
            continue
        tree = matches[0]
        if "data-paul" not in tree.path.parts:
            continue
        family = tree.path.parent.name
        lookups.append({
            "probe": probe,
            "tree": tree.path.stem,
            "family": family,
            "features": features,
            "output": actual,
            "width": shape[1],
        })

print(f"Unique Paul-tree matches: {len(lookups)}")
print(f"Unmatched or ambiguous lookups across all resources: {len(unmatched)}")
print(f"Paul tree files observed: {len({row['tree'] for row in lookups})}")

## Per-tree observations

This table reports the number of captured calls, tree output width, selector indices used by the parsed tree, and the number of distinct values observed at each selector. The slot numbers are raw positions in the runtime vector; they are not phonetic labels.

In [ ]:
tree_by_name = {tree.path.stem: tree for tree in trees if "data-paul" in tree.path.parts}
summary = defaultdict(list)
for row in lookups:
    summary[(row["family"], row["tree"])].append(row)

table_rows = []
for (family, name), rows in sorted(summary.items()):
    tree = tree_by_name[name]
    selectors = sorted({node.feature for node in tree.nodes})
    value_counts = [len({row['features'][slot] for row in rows}) for slot in selectors]
    selector_text = ", ".join(f"{slot}: {count}" for slot, count in zip(selectors, value_counts))
    outputs = [value for row in rows for value in row["output"]]
    table_rows.append((family, name, len(rows), tree.output_width, selector_text, min(outputs), max(outputs), len(set(outputs))))

headers = ("Family", "Tree", "Calls", "Width", "Selector: distinct inputs", "Output min", "Output max", "Distinct outputs")
thead = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
tbody = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in table_rows)
display(HTML(f"<table><thead><tr>{thead}</tr></thead><tbody>{tbody}</tbody></table>"))

## Selector coverage heatmap

Each cell shows how many distinct values appeared for that tree's selector across these captures. Empty cells mean the tree does not use that selector. Darker cells indicate greater observed variety, not greater importance. The captures are a small controlled sample, so unseen values and semantic interpretation remain open.

In [ ]:
MAX_SELECTORS = 12
CELL_W, CELL_H, LABEL_W, TOP_H = 38, 25, 115, 42
ordered = sorted(summary)
max_distinct = max((len({row['features'][slot] for row in rows})
                    for (family, name), rows in summary.items()
                    for slot in {node.feature for node in tree_by_name[name].nodes}), default=1)
width = LABEL_W + MAX_SELECTORS * CELL_W + 15
height = TOP_H + len(ordered) * CELL_H + 12
parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
parts.append('<style>text{font:12px sans-serif;fill:#202124}.axis{fill:#5f6368}</style>')
for slot in range(MAX_SELECTORS):
    x = LABEL_W + slot * CELL_W + CELL_W // 2
    parts.append(f'<text class="axis" x="{x}" y="25" text-anchor="middle">{slot}</text>')
for row_index, (family, name) in enumerate(ordered):
    rows = summary[(family, name)]
    tree = tree_by_name[name]
    y = TOP_H + row_index * CELL_H
    label = escape(family + chr(47) + name)
    parts.append(f'<text x="4" y="{y + 17}">{label}</text>')
    for slot in sorted({node.feature for node in tree.nodes}):
        distinct = len({row['features'][slot] for row in rows})
        strength = distinct / max_distinct
        red, green, blue = int(236 - 196 * strength), int(244 - 120 * strength), int(250 - 83 * strength)
        color = f"#{red:02x}{green:02x}{blue:02x}"
        x = LABEL_W + slot * CELL_W
        title = escape(f"{family}/{name}, selector {slot}: {distinct} distinct values across {len(rows)} calls")
        parts.append(f'<rect x="{x}" y="{y}" width="{CELL_W - 2}" height="{CELL_H - 2}" rx="3" fill="{color}"><title>{title}</title></rect>')
        parts.append(f'<text x="{x + (CELL_W - 2) / 2}" y="{y + 16}" text-anchor="middle">{distinct}</text>')
parts.append('</svg>')
display(SVG("".join(parts)))
print(f"Cell value = distinct observed inputs; color scale maximum = {max_distinct}.")

## Reading this evidence

The charts describe only values observed at the runtime lookup boundary in the three named traces. They can identify selectors worth probing next, such as slots that varied across more than one tree family or outputs that changed across probes. They cannot establish that a selector means a particular phone, boundary, or prosodic feature. Use the decompiler call sites and a controlled input that changes one candidate property at a time before proposing a semantic label.

The Markdown engine flowchart remains the overview of the recovered path. This notebook is a companion for inspecting captured numeric evidence; it does not duplicate or replace the written findings.